# Fine-tuning Qwen3 8B — Query Classifier Qadhya

Fine-tune Qwen3 8B avec LoRA (4-bit) pour classifier les requêtes juridiques tunisiennes AR/FR.

**4 dimensions de sortie :**
- `legal_branch` : civil | penal | commercial | travail | famille | immobilier | fiscal | procedure | autre
- `intent`       : definition | procedure | jurisprudence | template | consultation | hors_scope
- `language`     : ar | fr | mixed
- `rag_needed`   : true | false

**Pipeline :**
1. Charger dataset (gold + synthetic → ~1700 exemples)
2. Charger Qwen3-8B en 4-bit avec Unsloth
3. Appliquer LoRA (r=16)
4. Entraîner avec SFTTrainer
5. Exporter GGUF → Ollama

**Prérequis GPU :** minimum 16 GB VRAM (A100 40GB recommandé via Colab/RunPod)

```bash
# Installation (Colab ou env Python 3.10+)
pip install unsloth trl datasets transformers bitsandbytes
```

## 0. Installation

In [ ]:
# Installation Unsloth + dépendances
# Décommenter selon l'environnement

# Google Colab (CUDA 12.x) :
# !pip install unsloth "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# RunPod / GPU cloud standard :
# !pip install unsloth trl datasets transformers bitsandbytes accelerate

# Vérifier que CUDA est disponible
import torch
print(f"CUDA disponible : {torch.cuda.is_available()}")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun'}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 1. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION — Adapter selon votre setup
# =============================================================================

# Modèle base
BASE_MODEL = "Qwen/Qwen3-8B"          # Qwen3 8B Instruct
MAX_SEQ_LENGTH = 512                   # Requêtes courtes → 512 suffisant
DTYPE = None                           # Auto-detect (bfloat16 sur Ampere+)
LOAD_IN_4BIT = True                    # QLoRA 4-bit

# LoRA
LORA_RANK = 16                         # r=16 : bon compromis perf/mémoire
LORA_ALPHA = 16                        # alpha = rank (standard)
LORA_DROPOUT = 0.0                     # 0 recommandé par Unsloth pour QLoRA

# Entraînement
BATCH_SIZE = 4                         # Per-device batch size
GRAD_ACCUM = 4                         # Effective batch = 16
LEARNING_RATE = 2e-4
MAX_STEPS = 500                        # ~500 steps sur 1700 exemples ≈ 5 epochs
WARMUP_RATIO = 0.1
LR_SCHEDULER = "cosine"
SEED = 42

# Données
DATASET_PATH = "data/finetune/full-classifier-dataset.jsonl"  # Gold + Synthetic
VAL_SPLIT = 0.05                       # 5% validation

# Sauvegarde
OUTPUT_DIR = "models/qadhya-classifier-lora"  # Checkpoints LoRA
GGUF_PATH  = "models/qadhya-classifier-q4.gguf"  # Export Ollama

print("Configuration chargée ✅")

## 2. Charger le modèle (Unsloth + 4-bit)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Modèle chargé : {BASE_MODEL}")
print(f"Paramètres : {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

## 3. Appliquer LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Économise VRAM
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

# Afficher le ratio de paramètres entraînables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Paramètres entraînables : {trainable / 1e6:.1f}M / {total / 1e9:.1f}B ({100*trainable/total:.2f}%)")

## 4. Préparer le chat template (ChatML / Qwen3)

In [ ]:
from unsloth.chat_templates import get_chat_template

# Appliquer le template ChatML (natif Qwen3)
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",  # Qwen3 utilise ChatML
)

# Vérifier le template
test_messages = [
    {"role": "system",    "content": "Tu es un classificateur juridique."},
    {"role": "user",      "content": "Quel est le délai de prescription en matière civile ?"},
    {"role": "assistant", "content": '{"legal_branch":"civil","intent":"procedure","language":"fr","rag_needed":true}'},
]

formatted = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False
)
print("Exemple formaté :")
print(formatted)
print(f"\nNombre de tokens : {len(tokenizer.encode(formatted))}")

## 5. Charger et formater le dataset

In [ ]:
import json
from datasets import Dataset

# Charger le JSONL
def load_jsonl(path):
    entries = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries

raw_data = load_jsonl(DATASET_PATH)
print(f"Dataset chargé : {len(raw_data)} exemples")

# Vérifier la structure
sample = raw_data[0]
print(f"\nStructure : {list(sample.keys())}")
print(f"Conversations : {len(sample['conversations'])} tours")
print(f"\nExemple :")
for conv in sample['conversations']:
    print(f"  [{conv['role']}] {conv['content'][:80]}")

In [ ]:
from unsloth.chat_templates import standardize_sharegpt
from sklearn.model_selection import train_test_split

# Convertir en format Hugging Face Dataset
dataset = Dataset.from_list(raw_data)

# Normaliser le format des conversations
dataset = standardize_sharegpt(dataset)

# Split train/val
split = dataset.train_test_split(test_size=VAL_SPLIT, seed=SEED)
train_dataset = split['train']
eval_dataset  = split['test']

print(f"Train : {len(train_dataset)} exemples")
print(f"Val   : {len(eval_dataset)} exemples")

# Distribution des labels (vérification)
from collections import Counter
import re

branches = []
intents  = []
for ex in raw_data:
    assistant_msg = next(c['content'] for c in ex['conversations'] if c['role'] == 'assistant')
    try:
        lbl = json.loads(assistant_msg)
        branches.append(lbl.get('legal_branch', '?'))
        intents.append(lbl.get('intent', '?'))
    except:
        pass

print(f"\nDistribution legal_branch : {dict(Counter(branches))}")
print(f"Distribution intent       : {dict(Counter(intents))}")

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Formater les conversations avec le template
def formatting_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_func, batched=True)
eval_dataset  = eval_dataset.map(formatting_func, batched=True)

print("Exemple formaté (train[0]) :")
print(train_dataset[0]["text"][:400])

## 6. Entraînement (SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,   # Packing = plus efficace pour séquences courtes
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        evaluation_strategy="steps",
        eval_steps=50,
        save_steps=100,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type=LR_SCHEDULER,
        seed=SEED,
        output_dir=OUTPUT_DIR,
        report_to="none",     # Désactiver W&B (ajouter "wandb" si souhaité)
    ),
)

# Entraîner uniquement sur les réponses du modèle (pas le contexte)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print(f"Trainer configuré : {MAX_STEPS} steps, LR={LEARNING_RATE}, batch_eff={BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# Afficher l'utilisation mémoire avant entraînement
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
    max_memory  = round(gpu_stats.total_memory / 1024**3, 3)
    print(f"Mémoire GPU utilisée : {used_memory} / {max_memory} GB ({100*used_memory/max_memory:.1f}%)")

# LANCER L'ENTRAÎNEMENT
trainer_stats = trainer.train()

print(f"\n✅ Entraînement terminé")
print(f"   Durée : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Loss finale : {trainer_stats.metrics['train_loss']:.4f}")

if torch.cuda.is_available():
    used_memory_after = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
    print(f"   Mémoire max utilisée : {used_memory_after} GB")

## 7. Évaluation rapide

In [ ]:
# Mettre le modèle en mode inférence
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = """Tu es un classificateur expert de requêtes juridiques tunisiennes.
Analyse la question et retourne UNIQUEMENT un JSON avec ces 4 champs :
{"legal_branch": "civil|penal|commercial|travail|famille|immobilier|fiscal|procedure|autre",
 "intent": "definition|procedure|jurisprudence|template|consultation|hors_scope",
 "language": "ar|fr|mixed",
 "rag_needed": true|false}"""

def classify(question: str) -> dict:
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=100,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Décoder seulement les nouveaux tokens
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    try:
        return json.loads(response.strip())
    except:
        return {"error": response}

# Tests
test_questions = [
    "Quel est le délai de prescription de droit commun en matière civile en Tunisie ?",
    "كيف يتم تقديم طلب الطلاق أمام المحكمة التونسية ؟",
    "Quelles sont les conditions pour licencier un employé en Tunisie ?",
    "Bonjour, comment vas-tu ?",
    "ما هو الفصل 375 من المجلة الجزائية التونسية ؟",
    "Can you give me a contrat de bail modèle en tunisie ?",
]

print("Tests d'inférence :")
print("=" * 70)
for q in test_questions:
    result = classify(q)
    print(f"Q: {q[:60]}")
    print(f"→ {result}")
    print()

In [ ]:
# Évaluation sur le split de validation
correct = {"legal_branch": 0, "intent": 0, "language": 0, "rag_needed": 0}
total = 0

# Utiliser un sous-ensemble pour la vitesse
eval_sample = eval_dataset.select(range(min(50, len(eval_dataset))))

for ex in eval_sample:
    # Extraire la question (role=user) et le label attendu (role=assistant)
    convs = ex.get('conversations', [])
    question = next((c['content'] for c in convs if c['role'] == 'user'), None)
    expected_str = next((c['content'] for c in convs if c['role'] == 'assistant'), None)

    if not question or not expected_str:
        continue

    try:
        expected = json.loads(expected_str)
        predicted = classify(question)

        for key in correct:
            if predicted.get(key) == expected.get(key):
                correct[key] += 1
        total += 1
    except Exception as e:
        pass

print(f"Évaluation sur {total} exemples (validation set) :")
print("-" * 40)
for key, cnt in correct.items():
    print(f"  {key:15s} : {cnt}/{total} = {100*cnt/total:.1f}%")

## 8. Sauvegarder les poids LoRA

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sauvegarder les adaptateurs LoRA (léger — quelques centaines de MB)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ LoRA adapters sauvegardés dans : {OUTPUT_DIR}")
print(f"   Taille : {sum(os.path.getsize(os.path.join(r, f)) for r, d, files in os.walk(OUTPUT_DIR) for f in files) / 1e6:.1f} MB")

## 9. Export GGUF pour Ollama

In [ ]:
import os
os.makedirs("models", exist_ok=True)

# Export GGUF Q4_K_M (bon équilibre qualité/taille)
# Q4_K_M ≈ 4.8 GB pour 8B — compatible Ollama VPS 8GB RAM
model.save_pretrained_gguf(
    "models/qadhya-classifier",
    tokenizer,
    quantization_method="q4_k_m"
)

gguf_file = "models/qadhya-classifier-Q4_K_M.gguf"
if os.path.exists(gguf_file):
    size_gb = os.path.getsize(gguf_file) / 1e9
    print(f"✅ GGUF exporté : {gguf_file} ({size_gb:.1f} GB)")
else:
    # Chercher le fichier généré
    import glob
    files = glob.glob("models/*.gguf")
    print(f"Fichiers GGUF générés : {files}")

## 10. Déploiement sur Ollama (VPS Qadhya)

### Sur le VPS :

```bash
# 1. Transférer le GGUF sur le VPS
scp models/qadhya-classifier-Q4_K_M.gguf moncabinet-prod:/opt/moncabinet/models/

# 2. Créer le Modelfile Ollama
cat > /opt/moncabinet/models/Modelfile-qadhya-classifier << 'EOF'
FROM /opt/moncabinet/models/qadhya-classifier-Q4_K_M.gguf

SYSTEM """Tu es un classificateur expert de requêtes juridiques tunisiennes.
Analyse la question et retourne UNIQUEMENT un JSON avec ces 4 champs :
{\"legal_branch\": \"civil|penal|commercial|travail|famille|immobilier|fiscal|procedure|autre\",
 \"intent\": \"definition|procedure|jurisprudence|template|consultation|hors_scope\",
 \"language\": \"ar|fr|mixed\",
 \"rag_needed\": true|false}"""

PARAMETER temperature 0.1
PARAMETER num_predict 100
EOF

# 3. Créer le modèle Ollama
docker exec qadhya-nextjs ollama create qadhya-classifier -f /opt/moncabinet/models/Modelfile-qadhya-classifier

# 4. Tester
docker exec qadhya-nextjs ollama run qadhya-classifier "Quel est le délai de prescription civile ?"
```

### Dans `operations-config.ts` :

```typescript
// Avant:
'query-classification': {
  model: { provider: 'ollama', name: 'qwen3:8b' },
  ...
}

// Après:
'query-classification': {
  model: { provider: 'ollama', name: 'qadhya-classifier' },
  ...
}
```

In [ ]:
# Générer le Modelfile automatiquement
modelfile_content = '''FROM ./qadhya-classifier-Q4_K_M.gguf

SYSTEM """Tu es un classificateur expert de requêtes juridiques tunisiennes.
Analyse la question et retourne UNIQUEMENT un JSON avec ces 4 champs :
{\\"legal_branch\\": \\"civil|penal|commercial|travail|famille|immobilier|fiscal|procedure|autre\\",
 \\"intent\\": \\"definition|procedure|jurisprudence|template|consultation|hors_scope\\",
 \\"language\\": \\"ar|fr|mixed\\",
 \\"rag_needed\\": true|false}"""

PARAMETER temperature 0.1
PARAMETER num_predict 100
PARAMETER stop "<|im_end|>"
'''

with open('models/Modelfile-qadhya-classifier', 'w') as f:
    f.write(modelfile_content)

print("✅ Modelfile généré : models/Modelfile-qadhya-classifier")
print("\n📋 Commandes de déploiement :")
print("  scp models/qadhya-classifier-Q4_K_M.gguf moncabinet-prod:/opt/moncabinet/models/")
print("  scp models/Modelfile-qadhya-classifier moncabinet-prod:/opt/moncabinet/models/")
print("  ssh moncabinet-prod 'docker exec qadhya-nextjs ollama create qadhya-classifier -f /opt/moncabinet/models/Modelfile-qadhya-classifier'")

## Résumé des fichiers produits

| Fichier | Description |
|---------|-------------|
| `models/qadhya-classifier-lora/` | Adaptateurs LoRA (à conserver pour reprendre le training) |
| `models/qadhya-classifier-Q4_K_M.gguf` | Modèle quantisé pour Ollama (~4.8 GB) |
| `models/Modelfile-qadhya-classifier` | Config Ollama |

## Prochaines étapes

1. **Baseline** : comparer accuracy gold dataset avant/après fine-tuning
2. **AB test prod** : router 10% des queries vers `qadhya-classifier`, mesurer RAG Recall
3. **Itération** : ajouter plus d'exemples arabes si accuracy AR < 85%
4. **Monitoring** : logger les classifications erronées en prod pour amélioration continue